# Jordan-Normalform einer Matrix — mit Zwischenschritten

Dieses Notebook berechnet die **Jordan-Normalform (JNF)** einer beliebigen (quadratischen) Matrix $A$
und zeigt dabei **alle Zwischenschritte**, die in der Regelungstechnik relevant sind, z. B. bei der
Analyse von Zustandsraummodellen

$$
\dot{\mathbf{x}} = A\,\mathbf{x} + B\,u,\qquad y = C\,\mathbf{x} + D\,u,
$$

wenn $A$ nicht diagonalisierbar ist (mehrfache Eigenwerte mit geometrischer Vielfachheit < algebraischer
Vielfachheit). Die JNF ist dann die "beste mögliche" Diagonalform und liefert z. B. die modale Form
des Systems sowie Aussagen über die Stabilität und das Übergangsverhalten (Terme wie $t\,e^{\lambda t}$).

**Ablauf:**
1. Matrix $A$ definieren
2. Charakteristisches Polynom $\det(A-\lambda I)$ berechnen
3. Eigenwerte $\lambda_i$ und ihre algebraische Vielfachheit bestimmen
4. Für jeden Eigenwert: geometrische Vielfachheit (Anzahl der Jordan-Blöcke) über $\mathrm{rang}(A-\lambda_i I)$
5. Jordan-Kettenlängen über die Ränge von $(A-\lambda_i I)^k$ bestimmen
6. Eigenvektoren und Hauptvektoren (verallgemeinerte Eigenvektoren) berechnen
7. Transformationsmatrix $T$ aus den Vektorketten zusammensetzen
8. Jordan-Form $J = T^{-1} A T$ berechnen und verifizieren
9. Vergleich mit `sympy`'s eingebauter `jordan_form()`-Funktion

> Hinweis: Alle Rechnungen laufen **symbolisch** mit `sympy`, damit auch Matrizen mit Parametern
> (z. B. Reglerparametern) exakt behandelt werden können.


## 1. Imports

In [1]:
import sympy as sp
from sympy import Matrix, eye, zeros, symbols, simplify, latex
from IPython.display import display, Math

sp.init_printing()


## 2. Matrix $A$ definieren

Hier eine beliebige quadratische Matrix eintragen. Als Beispiel wird eine Matrix mit einem
mehrfachen Eigenwert verwendet, deren geometrische Vielfachheit kleiner als die algebraische ist
(typischer Fall, in dem die JNF *nicht* diagonal ist).

Einfach `A` durch die eigene Systemmatrix ersetzen (z. B. die $A$-Matrix eines Zustandsraummodells).


In [2]:
# Beispielmatrix -> hier durch die eigene (n x n) Matrix ersetzen
A = Matrix([
    [1, -4, 0],
    [2, 1, 0],
    [0, 0, 5],
])

n = A.shape[0]
display(Math(r"A = " + latex(A)))
print(f"Dimension: {n} x {n}")


<IPython.core.display.Math object>

Dimension: 3 x 3


## 3. Charakteristisches Polynom

$$
p(\lambda) = \det(A - \lambda I)
$$

Die Nullstellen von $p(\lambda)$ sind die Eigenwerte von $A$.


In [3]:
lam = symbols('lambda')
char_poly = sp.factor(A.charpoly(lam).as_expr())
display(Math(r"p(\lambda) = \det(A-\lambda I) = " + latex(char_poly)))


<IPython.core.display.Math object>

## 4. Eigenwerte und algebraische Vielfachheit

`eigenvals()` liefert ein Dictionary `{Eigenwert: algebraische Vielfachheit}`.


In [4]:
eigenvalues = A.eigenvals()

for lam_i, alg_mult in eigenvalues.items():
    display(Math(rf"\lambda = {latex(lam_i)}, \quad \text{{algebraische Vielfachheit }} m_a = {alg_mult}"))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 5. Geometrische Vielfachheit und Anzahl der Jordan-Blöcke

Die **geometrische Vielfachheit** eines Eigenwerts $\lambda_i$ ist die Dimension seines Eigenraums:

$$
m_g(\lambda_i) = \dim \ker(A - \lambda_i I) = n - \mathrm{rang}(A - \lambda_i I)
$$

Sie entspricht der **Anzahl der Jordan-Blöcke** zu diesem Eigenwert. Ist $m_g = m_a$, ist der Eigenwert
"regulär" und trägt nur $1\times 1$-Blöcke bei (im diagonalisierbaren Fall). Ist $m_g < m_a$, gibt es
mindestens einen Jordan-Block der Größe $> 1$.


In [5]:
geo_mult = {}
for lam_i in eigenvalues:
    M = A - lam_i * eye(n)
    rang = M.rank()
    mg = n - rang
    geo_mult[lam_i] = mg
    display(Math(
        rf"\lambda = {latex(lam_i)}: \ \mathrm{{rang}}(A-\lambda I) = {rang} "
        rf"\ \Rightarrow\ m_g = n - \mathrm{{rang}} = {mg} \ \text{{Jordan-Block(e)}}"
    ))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 6. Jordan-Kettenlängen bestimmen

Für jeden Eigenwert $\lambda_i$ werden die Ränge der Potenzen $(A-\lambda_i I)^k$ für
$k = 1, 2, \dots$ betrachtet, bis der Rangabfall stoppt (spätestens bei $k = m_a$).

Die Anzahl der Jordan-Blöcke der Größe $\geq k$ ergibt sich aus

$$
d_k = \mathrm{rang}\big((A-\lambda_i I)^{k-1}\big) - \mathrm{rang}\big((A-\lambda_i I)^{k}\big)
$$

(mit $(A-\lambda_i I)^0 = I$). Daraus lässt sich die exakte Blockstruktur (Partition der
algebraischen Vielfachheit) rekonstruieren.


In [6]:
block_structure = {}  # lambda -> Liste der Blockgroessen

for lam_i, alg_mult in eigenvalues.items():
    M = A - lam_i * eye(n)
    ranks = [n]  # rang((A-lam I)^0) = rang(I) = n
    Mk = eye(n)
    k = 0
    while True:
        k += 1
        Mk = Mk * M
        r = Mk.rank()
        ranks.append(r)
        if r == ranks[-2] or k >= alg_mult:
            break

    print(f"lambda = {lam_i}:")
    d_list = []
    for k in range(1, len(ranks)):
        d_k = ranks[k-1] - ranks[k]
        d_list.append(d_k)
        print(f"  rang((A-lambda I)^{k}) = {ranks[k]}   ->  d_{k} = {ranks[k-1]} - {ranks[k]} = {d_k} Block(e) der Groesse >= {k}")

    # Anzahl Bloecke exakter Groesse j = d_j - d_{j+1}
    d_list_ext = d_list + [0]
    sizes = []
    for j in range(len(d_list)):
        anzahl_genau_j = d_list_ext[j] - d_list_ext[j+1]
        sizes += [j+1] * anzahl_genau_j
    sizes.sort(reverse=True)
    block_structure[lam_i] = sizes
    print(f"  -> Blockgroessen zu lambda={lam_i}: {sizes}\n")


lambda = 1 - 2*sqrt(2)*I:
  rang((A-lambda I)^1) = 2   ->  d_1 = 3 - 2 = 1 Block(e) der Groesse >= 1
  -> Blockgroessen zu lambda=1 - 2*sqrt(2)*I: [1]

lambda = 1 + 2*sqrt(2)*I:
  rang((A-lambda I)^1) = 2   ->  d_1 = 3 - 2 = 1 Block(e) der Groesse >= 1
  -> Blockgroessen zu lambda=1 + 2*sqrt(2)*I: [1]

lambda = 5:
  rang((A-lambda I)^1) = 2   ->  d_1 = 3 - 2 = 1 Block(e) der Groesse >= 1
  -> Blockgroessen zu lambda=5: [1]



## 7. Eigenvektoren und Hauptvektoren (Jordan-Ketten)

Für jeden Jordan-Block der Größe $s$ zu $\lambda_i$ wird eine **Jordan-Kette**
$v_1, v_2, \dots, v_s$ gesucht mit

$$
(A-\lambda_i I)\,v_1 = 0 \quad\text{(gewöhnlicher Eigenvektor)}
$$
$$
(A-\lambda_i I)\,v_{k} = v_{k-1} \quad\text{für } k = 2,\dots,s \quad\text{(Hauptvektoren)}
$$

D. h. $v_s$ löst $(A-\lambda_i I)^s v_s = 0$, aber $(A-\lambda_i I)^{s-1} v_s \neq 0$.
Diese Ketten liefern gemeinsam die Spalten der Transformationsmatrix $T$.


In [7]:
def find_chain(A, lam_i, size, used_vectors, n):
    # Findet eine Jordan-Kette der Laenge `size` zu Eigenwert lam_i,
    # die linear unabhaengig von used_vectors ist.
    #
    # Konstruktionsprinzip: Ein "Hauptvektor der Stufe s" (Top der Kette)
    # ist ein Vektor v mit M^s v = 0, aber M^(s-1) v != 0 (M = A - lambda I).
    # Die restlichen Kettenglieder ergeben sich durch VORWAERTS-Multiplikation:
    #   v_s = v,  v_{s-1} = M v_s,  v_{s-2} = M v_{s-1}, ...,  v_1 = M^(s-1) v_s
    # (v_1 ist dann automatisch ein gewoehnlicher Eigenvektor, da M v_1 = M^s v = 0).
    M = A - lam_i * eye(n)
    Ms = M**size
    ns = Ms.nullspace()

    for cand in ns:
        # sicherstellen, dass die Kette wirklich die volle Laenge `size` hat,
        # d.h. cand darf NICHT bereits in ker(M^(size-1)) liegen
        if size > 1:
            if (M**(size - 1) * cand) == zeros(n, 1):
                continue
        else:
            if cand == zeros(n, 1):
                continue

        # Kette per Vorwaertsmultiplikation aufbauen: [v_size, v_{size-1}, ..., v_1]
        chain_top_down = [cand]
        v = cand
        for _ in range(size - 1):
            v = M * v
            chain_top_down.append(v)
        chain = list(reversed(chain_top_down))  # [v_1, v_2, ..., v_size]

        # Lineare Unabhaengigkeit von bereits verwendeten Vektoren pruefen
        test_basis = used_vectors + chain
        Mtest = Matrix.hstack(*test_basis)
        if Mtest.rank() == len(test_basis):
            return chain

    raise RuntimeError(f"Keine unabhaengige Kette der Laenge {size} zu lambda={lam_i} gefunden.")


jordan_chains = {}   # lambda -> Liste von Ketten (jede Kette: Liste von Matrix-Spaltenvektoren)

for lam_i, sizes in block_structure.items():
    used = []
    chains = []
    # groesste Bloecke zuerst bestimmen (stabiler)
    for size in sorted(sizes, reverse=True):
        chain = find_chain(A, lam_i, size, used, n)
        chains.append(chain)
        used.extend(chain)
    jordan_chains[lam_i] = chains

    print(f"lambda = {lam_i}: {len(chains)} Kette(n), Groessen {[len(c) for c in chains]}")
    for idx, chain in enumerate(chains, start=1):
        print(f"  Kette {idx} (Laenge {len(chain)}):")
        for j, v in enumerate(chain, start=1):
            label = "Eigenvektor v1" if j == 1 else f"Hauptvektor v{j}"
            display(Math(rf"\quad {label}: \ " + latex(v.T)))


lambda = 1 - 2*sqrt(2)*I: 1 Kette(n), Groessen [1]
  Kette 1 (Laenge 1):


<IPython.core.display.Math object>

lambda = 1 + 2*sqrt(2)*I: 1 Kette(n), Groessen [1]
  Kette 1 (Laenge 1):


<IPython.core.display.Math object>

lambda = 5: 1 Kette(n), Groessen [1]
  Kette 1 (Laenge 1):


<IPython.core.display.Math object>

## 8. Transformationsmatrix $T$ zusammensetzen

Alle Jordan-Ketten (aller Eigenwerte) werden spaltenweise zu $T$ zusammengefügt, in der Reihenfolge,
in der die zugehörigen Jordan-Blöcke später in $J$ erscheinen sollen.


In [8]:
T_columns = []
J_blocks = []

for lam_i, chains in jordan_chains.items():
    for chain in chains:
        T_columns.extend(chain)
        size = len(chain)
        Jb = lam_i * eye(size)
        for i in range(size - 1):
            Jb[i, i+1] = 1  # Superdiagonale
        J_blocks.append(Jb)

T = Matrix.hstack(*T_columns)
J = sp.diag(*J_blocks)
T_invert=T.inv()
display(Math(r"T = " + latex(T)))
display(Math(r"J = " + latex(J)))
display(Math(r"T^-1 = " + latex(T_invert)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 9. Verifikation

Es muss gelten: $A\,T = T\,J$, äquivalent $J = T^{-1} A T$.


In [9]:
AT = simplify(A * T)
TJ = simplify(T * J)

print("A*T == T*J :", AT.equals(TJ))

J_check = simplify(T.inv() * A * T)
display(Math(r"T^{-1} A T = " + latex(J_check)))


A*T == T*J : True


<IPython.core.display.Math object>

## 10. Vergleich mit `sympy`'s eingebauter Funktion

Zur Kontrolle: `Matrix.jordan_form()` liefert ebenfalls $(P, J)$ mit $A = P J P^{-1}$.
Die Blockgrößen sollten (bis auf Reihenfolge) mit dem oben hergeleiteten Ergebnis übereinstimmen.


In [10]:
P_sym, J_sym = A.jordan_form()
display(Math(r"J_{\text{sympy}} = " + latex(J_sym)))

print("Uebereinstimmung mit eigener Berechnung (bis auf Blockreihenfolge):",
      sp.simplify(A - P_sym * J_sym * P_sym.inv()) == zeros(n, n))


<IPython.core.display.Math object>

Uebereinstimmung mit eigener Berechnung (bis auf Blockreihenfolge): True


## 11. Anwendung: eigene Matrix einsetzen

Um die Analyse für eine eigene Zustandsmatrix $A$ (z. B. aus der Regelungstechnik) durchzuführen,
einfach in **Zelle 2** die Matrix `A` ersetzen und das Notebook erneut von oben nach unten ausführen
(*Kernel → Restart & Run All*). Alle Zwischenschritte (charakteristisches Polynom, Eigenwerte,
Vielfachheiten, Jordan-Ketten, $T$ und $J$) werden automatisch neu berechnet.

**Interpretation im Kontext der Regelungstechik:**
- Die Diagonaleinträge von $J$ sind die Eigenwerte des Systems (Pole) → Stabilität, falls alle
  $\mathrm{Re}(\lambda_i) < 0$.
- Ein Jordan-Block der Größe $s>1$ zu $\lambda_i$ führt in der Sprungantwort/Eigenbewegung zu
  Termen der Form $t^{k}\,e^{\lambda_i t}$ für $k=0,\dots,s-1$ statt nur $e^{\lambda_i t}$.
- Mit $\mathbf{x} = T\,\mathbf{z}$ transformiert man das System in die (Jordan-)Modalform
  $\dot{\mathbf{z}} = J\,\mathbf{z} + T^{-1}B\,u$.


## 10. Paket-Integration (Zustandsraum)

Dieser Abschnitt bündelt die Paketfunktionen für Zustandsraum- und Jordan-Themen.

- Eingang: Zustandsmatrix bzw. Übertragungsfunktion
- Ausgabe: Jordanform, Transition, Klassifikation und Poincare-Plot

In [ ]:
from pathlib import Path
from IPython.display import Image, display
from regelungstechnik import (
    zustandsraum_zu_uebertragungsfunktion,
    regelungsnormalform,
    transitionsmatrix,
    transitionsmatrix_symbolisch,
    jordan_normalform,
    poincare_klassifikation,
    plot_poincare,
)

# Eingabe
A_num = [[0, 1], [-2, -3]]
B_num = [[0], [1]]
C_num = [[1, 0]]
D_num = [[0]]

print("EINGABE A, B, C, D:")
print("A =", A_num)
print("B =", B_num)
print("C =", C_num)
print("D =", D_num)

# Ausgabe: Zustandsraum -> Uebertragungsfunktion
res_tf = zustandsraum_zu_uebertragungsfunktion(A_num, B_num, C_num, D_num)
num, den = res_tf["ergebnis"]
print("\nAUSGABE Uebertragungsfunktion:")
print("num =", num)
print("den =", den)

# Ausgabe: Regelungsnormalform
res_rnf = regelungsnormalform(num, den)
A_rnf, B_rnf, C_rnf, D_rnf = res_rnf["ergebnis"]
print("\nAUSGABE Regelungsnormalform (A, B, C, D):")
print(A_rnf)
print(B_rnf)
print(C_rnf)
print(D_rnf)

# Ausgabe: Transition (numerisch + symbolisch)
res_phi_num = transitionsmatrix(A_num, [0.0, 0.5, 1.0])
res_phi_sym = transitionsmatrix_symbolisch(A_num)
print("\nAUSGABE Transitionsmatrix numerisch:")
for idx, phi in enumerate(res_phi_num["ergebnis"]):
    print(f"t[{idx}] =")
    print(phi)
print("\nAUSGABE Transitionsmatrix symbolisch:")
display(res_phi_sym["ergebnis"])

# Ausgabe: Jordan + Poincare
res_j = jordan_normalform(A_num)
res_pk = poincare_klassifikation(A_num)
print("\nAUSGABE Jordanform J:")
display(res_j["ergebnis"]["J"])
print("AUSGABE Klassifikation:", res_pk["ergebnis"]["typ"])

plot_path = plot_poincare(A_num)
print("\nAUSGABE Poincare-Plot:", plot_path)
if Path(plot_path).exists():
    display(Image(filename=plot_path))